In [2]:
import pandas as pd
df = pd.read_csv("data/stock prices.csv", usecols=["date", "close", "high", "low", "open", "adjClose", "adjHigh", "adjLow", "adjOpen"]).dropna()
df.head()

,date,close,high,low,open,adjClose,adjHigh,adjLow,adjOpen
0,2015-05-27 00:00:00+00:00,132.045,132.260,130.05,130.34,121.682558,121.880685,119.844118,120.111360
1,2015-05-28 00:00:00+00:00,131.780,131.950,131.10,131.86,121.438354,121.595013,120.811718,121.512076
2,2015-05-29 00:00:00+00:00,130.280,131.450,129.90,131.23,120.056069,121.134251,119.705890,120.931516
3,2015-06-01 00:00:00+00:00,130.535,131.390,130.05,131.20,120.291057,121.078960,119.844118,120.903870
4,2015-06-02 00:00:00+00:00,129.960,130.655,129.32,129.86,119.761181,120.401640,119.171406,119.669029


In [3]:
df = df.sort_values(by="date").set_index(keys="date", drop=True)
df.head(15)

,close,high,low,open,adjClose,adjHigh,adjLow,adjOpen
date,,,,,,,,
2015-05-27 00:00:00+00:00,132.045,132.260,130.050,130.340,121.682558,121.880685,119.844118,120.111360
2015-05-28 00:00:00+00:00,131.780,131.950,131.100,131.860,121.438354,121.595013,120.811718,121.512076
2015-05-29 00:00:00+00:00,130.280,131.450,129.900,131.230,120.056069,121.134251,119.705890,120.931516
2015-06-01 00:00:00+00:00,130.535,131.390,130.050,131.200,120.291057,121.078960,119.844118,120.903870
2015-06-02 00:00:00+00:00,129.960,130.655,129.320,129.860,119.761181,120.401640,119.171406,119.669029
2015-06-03 00:00:00+00:00,130.120,130.940,129.900,130.660,119.908625,120.664274,119.705890,120.406248
2015-06-04 00:00:00+00:00,129.360,130.580,128.910,129.580,119.208267,120.332526,118.793582,119.411002
2015-06-05 00:00:00+00:00,128.650,129.690,128.360,129.500,118.553986,119.512370,118.286744,119.337280
2015-06-08 00:00:00+00:00,127.800,129.210,126.830,128.900,117.770691,119.070039,116.876813,118.784366


In [4]:
# Splitting the dataset
test_split = 0.2
train_df = df.iloc[ : int(df.shape[0] * (1 - test_split)), : ]
test_df = df.iloc[int(df.shape[0] * (1 - test_split)): , : ]

X_train, y_train = train_df.drop(columns=['close']), train_df['close']
X_test, y_test = test_df.drop(columns=['close']), test_df['close']

In [5]:
# Shapes
print(X_train.shape)
print(X_test.shape)

(1006, 7)
(252, 7)


In [6]:
# Scaling the input features
from sklearn.preprocessing import MinMaxScaler

scaler_x = MinMaxScaler(feature_range=(0, 1))
scaler_x.fit(X_train)

scaler_y = MinMaxScaler(feature_range=(0, 1))
scaler_y.fit(y_train.to_frame())

# Transforming train and test input features → NumPy Array
X_train = scaler_x.transform(X_train)
X_test = scaler_x.transform(X_test)

y_train = scaler_y.transform(y_train.to_frame())
y_test = scaler_y.transform(y_test.to_frame())

In [7]:
# Shapes
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(1006, 7)
(252, 7)
(1006, 1)
(252, 1)


In [8]:
# Dataset Preperation for Deep Learning model
import numpy as np

def prepare_dataset(dataset: np.array, timestep: int = 1) -> pd.DataFrame:
    dataset = pd.DataFrame(dataset)
    periods = list(range(1, timestep + 1))
    return dataset.shift(periods).iloc[timestep + 1:].to_numpy()

# timestep —→ Hyperparameter (More would be great)
timestep = 50
X_train = prepare_dataset(dataset=X_train, timestep=timestep)
X_test = prepare_dataset(dataset=X_test, timestep=timestep)

y_train = y_train[timestep + 1 : ]
y_test = y_test[timestep + 1 : ]

In [9]:
# Shapes
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(955, 350)
(201, 350)
(955, 1)
(201, 1)


In [10]:
# Model Building
import torch
from torch.nn import Module, RNN, LSTM, GRU, Linear

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [12]:
class CustomModel(Module):
    def __init__(self, input_size: int, mode: str="rnn"):
        super().__init__()
        self.mode = mode

        # sequential layers
        self.rnn = RNN(input_size=input_size, hidden_size=25, num_layers=1, nonlinearity="tanh", batch_first=True, bidirectional=False)
        self.lstm = LSTM(input_size=input_size, hidden_size=25, num_layers=1, batch_first=True, bidirectional=False)
        self.gru = GRU(input_size=input_size, hidden_size=25, num_layers=1, batch_first=True, bidirectional=False)

        self.linear = Linear(in_features=25, out_features=1)
        
    def forward(self, X_train):
        if self.mode == "rnn":
            _, final = self.rnn(X_train)
        elif self.mode == "lstm":
            _, (final, final_cell_state) = self.lstm(X_train)
        else:
            _, final = self.gru(X_train)

        y_pred = self.linear(final)
        return y_pred

In [13]:
# Training parameters
learning_rate = 0.001
epochs = 2500

In [14]:
# Loss function and optimizer
rnn_model = CustomModel(input_size=X_train.shape[1] // timestep, mode="rnn")
lstm_model = CustomModel(input_size=X_train.shape[1] // timestep, mode="lstm")
gru_model = CustomModel(input_size=X_train.shape[1] // timestep, mode="gru")

criterion = torch.nn.modules.loss.HuberLoss(delta=1.0)
rnn_optimizer = torch.optim.Adam(params = rnn_model.parameters())
lstm_optimizer = torch.optim.Adam(params = lstm_model.parameters())
gru_optimizer = torch.optim.Adam(params = gru_model.parameters())

In [16]:
rnn_model.to(device=device)
lstm_model.to(device=device)
gru_model.to(device=device)

CustomModel(
  (rnn): RNN(7, 25, batch_first=True)
  (lstm): LSTM(7, 25, batch_first=True)
  (gru): GRU(7, 25, batch_first=True)
  (linear): Linear(in_features=25, out_features=1, bias=True)
)

In [17]:
# Loss reduction storing
rnn_loss = []
lstm_loss = []
gru_loss = []

In [18]:
# Training pipeline for RNN model
for i in range(epochs):

    # Conterting data from NumPy array to Tensor
    X_train_cp = torch.tensor(X_train, dtype=torch.float32).reshape(X_train.shape[0], timestep, X_train.shape[1] // timestep).to(device) # NOTE —→ Hard coded
    y_train_cp = torch.tensor(y_train, dtype=torch.float32).to(device)

    # Initializing all gradients to zero
    rnn_optimizer.zero_grad()

    # Forward propogation
    y_pred = rnn_model(X_train_cp)

    # Loss calculation
    loss = criterion(y_pred.reshape(-1), y_train_cp.reshape(-1))
    rnn_loss.append(loss.item())

    # Gradient calculation
    loss.backward()

    # Updating weights
    rnn_optimizer.step()

    if (i + 1) % 100 == 0:
        print(f"Epoch {i + 1} → Loss = {loss.item()}")

Epoch 100 → Loss = 0.007289134431630373
Epoch 200 → Loss = 0.006306999363005161
Epoch 300 → Loss = 0.005108512006700039
Epoch 400 → Loss = 0.0040004379115998745
Epoch 500 → Loss = 0.003949059173464775
Epoch 600 → Loss = 0.006037476472556591
Epoch 700 → Loss = 0.003049815073609352
Epoch 800 → Loss = 0.0026991809718310833
Epoch 900 → Loss = 0.003426363691687584
Epoch 1000 → Loss = 0.003850245149806142
Epoch 1100 → Loss = 0.0028719671536237
Epoch 1200 → Loss = 0.0015722891548648477
Epoch 1300 → Loss = 0.001818622462451458
Epoch 1400 → Loss = 0.0014696152647957206
Epoch 1500 → Loss = 0.0014437958598136902
Epoch 1600 → Loss = 0.0014518835814669728
Epoch 1700 → Loss = 0.001361764851026237
Epoch 1800 → Loss = 0.0012280604569241405
Epoch 1900 → Loss = 0.0030174474231898785
Epoch 2000 → Loss = 0.001332704210653901
Epoch 2100 → Loss = 0.000982551951892674
Epoch 2200 → Loss = 0.0009669420542195439
Epoch 2300 → Loss = 0.0010591816389933228
Epoch 2400 → Loss = 0.0008336523897014558
Epoch 2500 → Los

In [19]:
rnn_model.eval()

CustomModel(
  (rnn): RNN(7, 25, batch_first=True)
  (lstm): LSTM(7, 25, batch_first=True)
  (gru): GRU(7, 25, batch_first=True)
  (linear): Linear(in_features=25, out_features=1, bias=True)
)

In [ ]:
with torch.no_grad():
    # Conterting data from NumPy array to Tensor
    X_test_cp = torch.tensor(X_test, dtype=torch.float32).reshape(X_test.shape[0], timestep, X_test.shape[1] // timestep).to(device) # NOTE —→ Hard coded
    y_test_cp = torch.tensor(y_test, dtype=torch.float32).to(device)

    # Forward propogation
    y_pred_rnn = rnn_model(X_test_cp)  

In [21]:
# Training pipeline for RNN model
for i in range(epochs):

    # Conterting data from NumPy array to Tensor
    X_train_cp = torch.tensor(X_train, dtype=torch.float32).reshape(X_train.shape[0], timestep, X_train.shape[1] // timestep).to(device) # NOTE —→ Hard coded
    y_train_cp = torch.tensor(y_train, dtype=torch.float32).to(device)

    # Initializing all gradients to zero
    lstm_optimizer.zero_grad()

    # Forward propogation
    y_pred = lstm_model(X_train_cp)

    # Loss calculation
    loss = criterion(y_pred.reshape(-1), y_train_cp.reshape(-1))
    lstm_loss.append(loss.item())

    # Gradient calculation
    loss.backward()

    # Updating weights
    lstm_optimizer.step()

    if (i + 1) % 100 == 0:
        print(f"Epoch {i + 1} → Loss = {loss.item()}")

Epoch 100 → Loss = 0.006433377042412758
Epoch 200 → Loss = 0.005586002953350544
Epoch 300 → Loss = 0.005453982390463352
Epoch 400 → Loss = 0.00526337418705225
Epoch 500 → Loss = 0.0038747028447687626
Epoch 600 → Loss = 0.000951899797655642
Epoch 700 → Loss = 0.0006822809809818864
Epoch 800 → Loss = 0.0005034386413171887
Epoch 900 → Loss = 0.0005851793102920055
Epoch 1000 → Loss = 0.0004095674667041749
Epoch 1100 → Loss = 0.00034951037378050387
Epoch 1200 → Loss = 0.0003694048791658133
Epoch 1300 → Loss = 0.0003097500593867153
Epoch 1400 → Loss = 0.00029750209068879485
Epoch 1500 → Loss = 0.00028802125598303974
Epoch 1600 → Loss = 0.0002804268733598292
Epoch 1700 → Loss = 0.00027186909574083984
Epoch 1800 → Loss = 0.00026649111532606184
Epoch 1900 → Loss = 0.0002622694300953299
Epoch 2000 → Loss = 0.00025676455697976053
Epoch 2100 → Loss = 0.0002538453263696283
Epoch 2200 → Loss = 0.00025301906862296164
Epoch 2300 → Loss = 0.00025017053121700883
Epoch 2400 → Loss = 0.0002439996605971828

In [23]:
lstm_model.eval()

CustomModel(
  (rnn): RNN(7, 25, batch_first=True)
  (lstm): LSTM(7, 25, batch_first=True)
  (gru): GRU(7, 25, batch_first=True)
  (linear): Linear(in_features=25, out_features=1, bias=True)
)

In [25]:
with torch.no_grad():
    # Conterting data from NumPy array to Tensor
    X_test_cp = torch.tensor(X_test, dtype=torch.float32).reshape(X_test.shape[0], timestep, X_test.shape[1] // timestep).to(device) # NOTE —→ Hard coded
    y_test_cp = torch.tensor(y_test, dtype=torch.float32).to(device)

    # Forward propogation
    y_pred_lstm = lstm_model(X_test_cp)  

In [26]:
# Training pipeline for RNN model
for i in range(epochs):

    # Conterting data from NumPy array to Tensor
    X_train_cp = torch.tensor(X_train, dtype=torch.float32).reshape(X_train.shape[0], timestep, X_train.shape[1] // timestep).to(device) # NOTE —→ Hard coded
    y_train_cp = torch.tensor(y_train, dtype=torch.float32).to(device)

    # Initializing all gradients to zero
    gru_optimizer.zero_grad()

    # Forward propogation
    y_pred = gru_model(X_train_cp)

    # Loss calculation
    loss = criterion(y_pred.reshape(-1), y_train_cp.reshape(-1))
    gru_loss.append(loss.item())

    # Gradient calculation
    loss.backward()

    # Updating weights
    gru_optimizer.step()

    if (i + 1) % 100 == 0:
        print(f"Epoch {i + 1} → Loss = {loss.item()}")

Epoch 100 → Loss = 0.007013502065092325
Epoch 200 → Loss = 0.004228097852319479
Epoch 300 → Loss = 0.003139608772471547
Epoch 400 → Loss = 0.0012839784612879157
Epoch 500 → Loss = 0.0005947084282524884
Epoch 600 → Loss = 0.0003816886164713651
Epoch 700 → Loss = 0.0002837276551872492
Epoch 800 → Loss = 0.00023596224491484463
Epoch 900 → Loss = 0.00021153950365260243
Epoch 1000 → Loss = 0.00019851750403176993
Epoch 1100 → Loss = 0.00019085027452092618
Epoch 1200 → Loss = 0.0001857143797678873
Epoch 1300 → Loss = 0.00018213491421192884
Epoch 1400 → Loss = 0.00017823492817115039
Epoch 1500 → Loss = 0.0001751983945723623
Epoch 1600 → Loss = 0.0001724917092360556
Epoch 1700 → Loss = 0.00017008066060952842
Epoch 1800 → Loss = 0.0001679603010416031
Epoch 1900 → Loss = 0.0001662219874560833
Epoch 2000 → Loss = 0.0001653585204621777
Epoch 2100 → Loss = 0.00016283792501781136
Epoch 2200 → Loss = 0.0001615123328519985
Epoch 2300 → Loss = 0.00016511073044966906
Epoch 2400 → Loss = 0.000159165065269

In [27]:
gru_model.eval()

CustomModel(
  (rnn): RNN(7, 25, batch_first=True)
  (lstm): LSTM(7, 25, batch_first=True)
  (gru): GRU(7, 25, batch_first=True)
  (linear): Linear(in_features=25, out_features=1, bias=True)
)

In [32]:
with torch.no_grad():
    # Conterting data from NumPy array to Tensor
    X_test_cp = torch.tensor(X_test, dtype=torch.float32).reshape(X_test.shape[0], timestep, X_test.shape[1] // timestep).to(device) # NOTE —→ Hard coded
    y_test_cp = torch.tensor(y_test, dtype=torch.float32).to(device)

    # Forward propogation
    y_pred_gru = gru_model(X_test_cp)  

In [39]:
import plotly.graph_objects as go
import numpy as np

# Ensure data is 1D arrays for Plotly
# If y_test or y_pred are 2D (e.g., shape (n_samples, 1)), flatten them
actual_prices = scaler_y.inverse_transform(y_test).flatten()

predicted_prices_rnn = scaler_y.inverse_transform(y_pred_rnn.reshape(1, -1).to('cpu').detach().numpy()).flatten()
predicted_prices_lstm = scaler_y.inverse_transform(y_pred_lstm.reshape(1, -1).to('cpu').detach().numpy()).flatten()
predicted_prices_gru = scaler_y.inverse_transform(y_pred_gru.reshape(1, -1).to('cpu').detach().numpy()).flatten()

# Create the x-axis (Days). 
# Assuming len(actual_prices) represents the number of days.
days = np.arange(len(actual_prices))

# Create the figure
fig = go.Figure()

# Add Actual Prices trace
fig.add_trace(go.Scatter(
    x=days,
    y=actual_prices,
    mode='lines',
    name='Actual Stock Prices',
    line=dict(color='blue', width=2)
))

# Add Predicted Prices trace
fig.add_trace(go.Scatter(
    x=days,
    y=predicted_prices_rnn,
    mode='lines',
    name='Predicted Stock Prices by RNN',
    line=dict(color='yellow')
))
fig.add_trace(go.Scatter(
    x=days,
    y=predicted_prices_lstm,
    mode='lines',
    name='Predicted Stock Prices by LSTM',
    line=dict(color='green', width=1)
))
fig.add_trace(go.Scatter(
    x=days,
    y=predicted_prices_gru,
    mode='lines',
    name='Predicted Stock Prices by GRU',
    line=dict(color='red', width=1)
))

# Update layout
fig.update_layout(
    title='Stock Price Prediction',
    xaxis_title='Days',
    yaxis_title='Price',
    hovermode='x unified',
    template='plotly_white',
    width=1500,  # Adjust width to match your matplotlib figsize preference
    height=700
)

# Show the chart
fig.show()

In [42]:
# Create the figure
fig = go.Figure()

# Add Predicted Prices trace
fig.add_trace(go.Scatter(
    x=list(range(len((rnn_loss)))),
    y=rnn_loss,
    mode='lines',
    name='RNN Loss',
    line=dict(color='red')
))

fig.add_trace(go.Scatter(
    x=list(range(len((lstm_loss)))),
    y=lstm_loss,
    mode='lines',
    name='LSTM Loss',
    line=dict(color='green', width=1)
))

fig.add_trace(go.Scatter(
    x=list(range(len((gru_loss)))),
    y=gru_loss,
    mode='lines',
    name='GRU Loss',
    line=dict(color='blue', width=1)
))

# Update layout
fig.update_layout(
    title='Loss comparison',
    xaxis_title='Epochs',
    yaxis_title='Loss',
    hovermode='x unified',
    template='plotly_white',
    width=1500,  # Adjust width to match your matplotlib figsize preference
    height=700
)

# Show the chart
fig.show()